# LSTM Timeseries Anomaly Detection - Public Workflow

This notebook provides a clean public version of the time-series anomaly detection workflow. It shows the sequence-preparation and anomaly-review logic without uploading cluttered experimental notebook setup.

## Project Goal

Use time-series thinking to identify unusual periods in sequential activity data and support model/analyst review.

In [ ]:
import pandas as pd

## Load Public Sample

The public sample contains timestamped NYC taxi activity values. The working sample used for output summaries contains 10,320 half-hourly records.

In [ ]:
data = pd.read_csv('../data/public_sample_nyc_taxi_timeseries.csv', parse_dates=['timestamp'])
data = data.sort_values('timestamp').reset_index(drop=True)
data.head()

## Create Sequence Windows

LSTM-style models use ordered windows of previous observations, which helps the model learn recent behaviour instead of treating each row as independent.

In [ ]:
def create_sequences(values, window_size=48):
    return [values[start:start + window_size] for start in range(len(values) - window_size)]

sequences = create_sequences(data['value'].to_numpy())
len(sequences)

## Build Review Candidates

A rolling baseline is used as a transparent review layer. It highlights points that differ most from recent behaviour and helps validate anomaly model output.

In [ ]:
reviewed = data.copy()
reviewed['rolling_mean_48'] = reviewed['value'].rolling(48, min_periods=12).mean()
reviewed['rolling_std_48'] = reviewed['value'].rolling(48, min_periods=12).std()
reviewed['deviation_score'] = ((reviewed['value'] - reviewed['rolling_mean_48']) / reviewed['rolling_std_48']).abs()
reviewed.dropna().sort_values('deviation_score', ascending=False).head(10)

## Output Review

The repository includes exported outputs for dataset profile, daily volume, and top anomaly review candidates. These files make the project easier to review directly on GitHub.